# Patent phrase matching: submission notebook

This notebook contains the full modeling workflow for the mini project described in [project_brief.md](project_brief.md): the fine-tuned cross-encoder baseline, an unfine-tuned bi-encoder comparison, two controlled context experiments, validation error analysis, and competition submission generation.

It includes the baseline comparison, controlled context experiments, and error-analysis tables. Run it on Kaggle to produce the evidence for your report.

**Running it**

- Kaggle notebook, **GPU T4 or P100**, attached inputs:
  1. the competition data, via **Add Input → Competitions → U.S. Patent Phrase to Phrase Matching**;
  2. the DeBERTa-v3-small and MiniLM checkpoint outputs from [download_weights.ipynb](download_weights.ipynb) (run it once for each `MODEL_ID`).
- Turn **Internet off** before you submit, because the competition runs submitted notebooks offline. The notebook falls back to the Hugging Face Hub only when it cannot find a local checkpoint, which works only with internet on.
- Selects a setting using the fixed validation split, then refits that cross-encoder on all labeled training rows before predicting the hidden test set. Writes `/kaggle/working/submission.csv` and `/kaggle/working/val_predictions.csv`.
- Uses packages preinstalled on Kaggle (`torch`, `transformers`, `datasets`, `scikit-learn`, `pandas`).

## 1. Setup

In [ ]:
import glob
import os

import numpy as np
import pandas as pd
import torch
import transformers
from datasets import Dataset
from sklearn.model_selection import GroupShuffleSplit
from tokenizers.processors import TemplateProcessing
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, Trainer, TrainingArguments)

SEED = 42
transformers.set_seed(SEED)   # seeds Python, NumPy, and PyTorch; GPU results can still vary slightly between runs
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| transformers", transformers.__version__, "| device:", device)

## 2. Configuration

These are the settings you will change in your experiments. Change **one** setting at a time so you can tell what caused a difference.

In [ ]:
MODEL_ID = "microsoft/deberta-v3-small"   # fine-tuned cross-encoder
BI_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"   # pretrained sentence-embedding baseline
MAX_LEN = 64            # anchor + target + context fit easily within 64 tokens
EPOCHS = 3
LR = 2e-5
BATCH_SIZE = 32
VAL_FRACTION = 0.2      # share of *anchors* (not rows) held out for validation
OUT_DIR = "/kaggle/working"


def find_one(pattern):
    """Return the directory of the first file under /kaggle/input that matches the pattern."""
    hits = glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)
    return os.path.dirname(hits[0]) if hits else None


DATA_DIR = find_one("us-patent-phrase-to-phrase-matching/train.csv") or find_one("train.csv")
assert DATA_DIR, "Competition data not found: add it with Add Input -> Competitions."
MODEL_PATH = find_one(f"{MODEL_ID.split('/')[-1]}/config.json") or MODEL_ID
BI_MODEL_PATH = find_one(f"{BI_MODEL_ID.split('/')[-1]}/config.json") or BI_MODEL_ID
print("data:", DATA_DIR)
print("model:", MODEL_PATH, "(local)" if MODEL_PATH != MODEL_ID else "(Hub download: needs internet)")
print("bi-encoder:", BI_MODEL_PATH, "(local)" if BI_MODEL_PATH != BI_MODEL_ID else "(Hub download: needs internet)")

## 3. Data

Each row asks how similar a **target** phrase is to an **anchor** phrase, judged within a patent **context**. The context is a CPC (Cooperative Patent Classification) code such as `G02`, whose first letter is the section. Scores are 0, 0.25, 0.5, 0.75, or 1. The visible `test.csv` has only a few rows; when you submit, Kaggle reruns the notebook with the hidden test set in its place.

In [ ]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
print("train:", train.shape, "| test (visible):", test.shape)
print(train.score.value_counts().sort_index())
train.head()

## 4. Model input

The model reads the anchor as the first text segment, and the target plus a short context description as the second. The tokenizer joins them as `[CLS] anchor [SEP] target; context [SEP]`, so attention can compare the two phrases token by token: this is what makes it a cross-encoder. Here the context is just the CPC section name. How you represent context is one of the experiments you can run.

In [ ]:
CPC_SECTIONS = {
    "A": "human necessities", "B": "performing operations; transporting",
    "C": "chemistry; metallurgy", "D": "textiles; paper", "E": "fixed constructions",
    "F": "mechanical engineering; lighting; heating; weapons; blasting",
    "G": "physics", "H": "electricity",
}


def add_inputs(df):
    df = df.copy()
    df["text_a"] = df["anchor"]
    df["text_b"] = df["target"] + "; " + df["context"].str[0].map(CPC_SECTIONS)
    return df


train, test = add_inputs(train), add_inputs(test)
train[["text_a", "text_b", "score"]].head()

## 5. Validation split

The training set contains only about 730 distinct anchors, each paired with many targets. If the same anchor appeared in both the training and validation sets, the validation score would overestimate performance on unseen anchors. The split therefore holds out whole **anchors**.

Keep this split fixed (same `SEED` and `VAL_FRACTION`) for every experiment, so your comparisons are fair.

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=VAL_FRACTION, random_state=SEED)
train_idx, val_idx = next(splitter.split(train, groups=train["anchor"]))
trn, val = train.iloc[train_idx].reset_index(drop=True), train.iloc[val_idx].reset_index(drop=True)
assert set(trn.anchor).isdisjoint(val.anchor)
print(f"train rows {len(trn)} | val rows {len(val)} | val anchors {val.anchor.nunique()}")

## 6. Tokenization

Padding is left to the data collator, which pads each batch only to its longest sequence. The printed example should start with `[CLS]` and have a `[SEP]` between the two segments.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
# transformers 5.0 builds this tokenizer without [CLS]/[SEP]; set the pair template explicitly
tokenizer.backend_tokenizer.post_processor = TemplateProcessing(
    single="[CLS] $A [SEP]", pair="[CLS] $A [SEP] $B:1 [SEP]:1",
    special_tokens=[("[CLS]", tokenizer.cls_token_id), ("[SEP]", tokenizer.sep_token_id)])


def to_dataset(df, with_labels=True):
    cols = ["text_a", "text_b"] + (["score"] if with_labels else [])
    ds = Dataset.from_pandas(df[cols].astype({"score": "float32"}) if with_labels else df[cols])  # MSE loss needs float32 labels
    ds = ds.map(lambda b: tokenizer(b["text_a"], b["text_b"], truncation=True, max_length=MAX_LEN),
                batched=True, remove_columns=["text_a", "text_b"])
    return ds.rename_column("score", "labels") if with_labels else ds


ds_trn, ds_val, ds_test = to_dataset(trn), to_dataset(val), to_dataset(test, with_labels=False)
print(tokenizer.decode(ds_trn[0]["input_ids"]))

## 7. Fine-tuning

With `num_labels=1`, the classification head outputs a single number. Because the labels are floats, Hugging Face treats the task as **regression** and minimizes mean squared error. The competition metric is the **Pearson correlation** between predictions and true scores. Pearson correlation ignores any linear rescaling of the predictions, so the model is rewarded for ranking pairs in the right order, not for hitting the exact values.

In [ ]:
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    return {"pearson": float(np.corrcoef(preds.reshape(-1), labels)[0, 1])}



# Load in fp32: the deberta-v3 checkpoints are stored in fp16, and fp16=True mixed precision below needs
# fp32 weights ("Attempting to unscale FP16 gradients" otherwise). Older transformers call this torch_dtype.
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH, num_labels=1, dtype=torch.float32)
args = TrainingArguments(
    output_dir=f"{OUT_DIR}/trainer_tmp",
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=2 * BATCH_SIZE,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",          # keeps disk use and runtime low; the final weights stay in memory
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
 )
trainer = Trainer(model=model, args=args, train_dataset=ds_trn, eval_dataset=ds_val,
                  data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=compute_metrics)
baseline_started = __import__("time").perf_counter()
trainer.train()
baseline_minutes = (__import__("time").perf_counter() - baseline_started) / 60
print(f"baseline training time: {baseline_minutes:.2f} minutes")

## 8. Validation predictions

The predictions are saved so you can do your error analysis without retraining.

In [ ]:
val["pred"] = trainer.predict(ds_val).predictions.reshape(-1)
val_pearson = np.corrcoef(val["pred"], val["score"])[0, 1]
print(f"Validation Pearson: {val_pearson:.4f}")
val[["id", "anchor", "target", "context", "score", "pred"]].to_csv(f"{OUT_DIR}/val_predictions.csv", index=False)

## 9. Submission

The competition expects `/kaggle/working/submission.csv` with columns `id,score` and one row per test row. Clipping to [0, 1] keeps the scores in the label range; it does not change the validation Pearson correlation much.

In [ ]:
baseline_test_pred = trainer.predict(ds_test).predictions.reshape(-1)
submission = pd.DataFrame({"id": test["id"], "score": np.clip(baseline_test_pred, 0, 1)})
assert len(submission) == len(test) and submission["score"].notna().all()
submission.to_csv(f"{OUT_DIR}/submission.csv", index=False)
submission.head()

del trainer, model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Experiments and final submission

The cells below compute the local validation baseline comparison, two controlled context experiments, and validation error analysis. Run the notebook on Kaggle with the competition data and attached checkpoints to record your actual results. The late submission and leaderboard comparison still require your Kaggle account.

1. **Baseline without fine-tuning**, evaluated on the same validation split.
2. **At least two controlled experiments**, each changing one setting from Section 2 or Section 4.
3. **Error analysis** using `val_predictions.csv`.
4. **Late submission** of your final notebook, then record its public and private leaderboard scores.

Before you submit, make sure the notebook runs from top to bottom with internet off. Delete or skip experiment cells the final submission does not need, so it stays within the competition's runtime limit.

## 10. Bi-encoder baseline and controlled context experiments

Run this section after the starter's fine-tuning cell. It reuses the same anchor-grouped `trn`/`val` split. The bi-encoder uses pretrained `sentence-transformers/all-MiniLM-L6-v2`, mean-pools each phrase separately, and predicts cosine similarity without training on these labels. Attach its checkpoint using [download_weights.ipynb](download_weights.ipynb). The two cross-encoder experiments change only the target-side context representation: no context, then the raw CPC code.

A bi-encoder processes each phrase independently, so cosine compares fixed phrase vectors; the fine-tuned cross-encoder sees both phrases together and can use token-level attention across them. Treat validation differences below about 0.005 as inconclusive unless repeated with another seed.

In [ ]:
from transformers import AutoModel


@torch.no_grad()
def mean_pool_embeddings(encoder, encoder_tokenizer, texts, batch_size=32):
    encoder.eval()
    vectors = []
    for start in range(0, len(texts), batch_size):
        batch = encoder_tokenizer(texts[start:start + batch_size], padding=True, truncation=True,
                          max_length=MAX_LEN, return_tensors="pt").to(device)
        hidden = encoder(**batch).last_hidden_state
        mask = batch["attention_mask"].unsqueeze(-1).to(hidden.dtype)
        vectors.append((hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1))
    return torch.cat(vectors, dim=0)


bi_tokenizer = AutoTokenizer.from_pretrained(BI_MODEL_PATH)
bi_encoder = AutoModel.from_pretrained(BI_MODEL_PATH).to(device).eval()
start_time = __import__("time").perf_counter()
anchor_vectors = mean_pool_embeddings(bi_encoder, bi_tokenizer, val["anchor"].tolist())
target_vectors = mean_pool_embeddings(bi_encoder, bi_tokenizer, val["target"].tolist())
bi_predictions = torch.nn.functional.cosine_similarity(anchor_vectors, target_vectors).cpu().numpy()
test_anchor_vectors = mean_pool_embeddings(bi_encoder, bi_tokenizer, test["anchor"].tolist())
test_target_vectors = mean_pool_embeddings(bi_encoder, bi_tokenizer, test["target"].tolist())
bi_test_predictions = torch.nn.functional.cosine_similarity(
    test_anchor_vectors, test_target_vectors).cpu().numpy()
bi_pearson = float(np.corrcoef(bi_predictions, val["score"].to_numpy())[0, 1])
bi_minutes = (__import__("time").perf_counter() - start_time) / 60
baseline_comparison = pd.DataFrame([
    {"model": "Unfine-tuned MiniLM bi-encoder", "validation Pearson": bi_pearson,
     "time (minutes)": bi_minutes, "time type": "encoding"},
    {"model": "Fine-tuned DeBERTa cross-encoder", "validation Pearson": val_pearson,
     "time (minutes)": baseline_minutes, "time type": "training"},
])
display(baseline_comparison)

In [ ]:
def run_context_experiment(mode):
    def apply_mode(frame):
        frame = frame.copy()
        if mode == "no context":
            frame["text_b"] = frame["target"]
        elif mode == "raw CPC code":
            frame["text_b"] = frame["target"] + "; " + frame["context"].astype(str)
        else:
            raise ValueError(mode)
        return frame

    experiment_train, experiment_val = apply_mode(trn), apply_mode(val)
    train_data = to_dataset(experiment_train)
    val_data = to_dataset(experiment_val)
    transformers.set_seed(SEED)
    experiment_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_PATH, num_labels=1, dtype=torch.float32)
    experiment_args = TrainingArguments(
        output_dir=f"{OUT_DIR}/context_{mode.replace(' ', '_')}",
        num_train_epochs=EPOCHS, learning_rate=LR,
        per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=2 * BATCH_SIZE,
        warmup_ratio=0.1, weight_decay=0.01, eval_strategy="epoch",
        save_strategy="no", logging_steps=100, fp16=torch.cuda.is_available(),
        report_to="none", seed=SEED)
    experiment_trainer = Trainer(
        model=experiment_model, args=experiment_args, train_dataset=train_data,
        eval_dataset=val_data, data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=compute_metrics)
    started = __import__("time").perf_counter()
    experiment_trainer.train()
    duration = (__import__("time").perf_counter() - started) / 60
    predictions = experiment_trainer.predict(val_data).predictions.reshape(-1)
    pearson = float(np.corrcoef(predictions, experiment_val["score"])[0, 1])
    context_predictions[mode] = predictions
    del experiment_trainer, experiment_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {"setting": mode, "validation Pearson": pearson, "training minutes": duration}


context_predictions = {}
context_results = [
    {"setting": "starter: CPC section title", "validation Pearson": val_pearson,
     "training minutes": baseline_minutes},
    run_context_experiment("no context"),
    run_context_experiment("raw CPC code"),
]
context_results = pd.DataFrame(context_results)
def interpret_context_result(row):
    delta = row["validation Pearson"] - val_pearson
    if row["setting"] == "starter: CPC section title":
        return "Reference setting."
    if abs(delta) < 0.005:
        return f"Change {delta:+.4f}; within ~0.005, treat as inconclusive without another seed."
    return f"Change {delta:+.4f} vs. reference; repeat with another seed before a strong conclusion."


context_results["interpretation"] = context_results.apply(interpret_context_result, axis=1)
display(context_results)

final_candidates = pd.concat([
    context_results[["setting", "validation Pearson"]],
    pd.DataFrame([{"setting": "Unfine-tuned MiniLM bi-encoder",
                   "validation Pearson": bi_pearson}]),
], ignore_index=True)
best_setting = final_candidates.loc[
    final_candidates["validation Pearson"].idxmax(), "setting"]
if best_setting == "Unfine-tuned MiniLM bi-encoder":
    best_val_pred = bi_predictions
else:
    best_val_pred = (val["pred"].to_numpy() if best_setting == "starter: CPC section title"
                     else context_predictions[best_setting])

val["pred"] = best_val_pred
val[["id", "anchor", "target", "context", "score", "pred"]].to_csv(
    f"{OUT_DIR}/val_predictions.csv", index=False)

# Choose by held-out validation score, then fit the chosen cross-encoder on all labeled rows.
if best_setting == "Unfine-tuned MiniLM bi-encoder":
    final_test_pred = bi_test_predictions
    final_fit_minutes = 0.0
else:
    final_train, final_test = train.copy(), test.copy()
    if best_setting == "no context":
        final_train["text_b"] = final_train["target"]
        final_test["text_b"] = final_test["target"]
    elif best_setting == "raw CPC code":
        final_train["text_b"] = final_train["target"] + "; " + final_train["context"].astype(str)
        final_test["text_b"] = final_test["target"] + "; " + final_test["context"].astype(str)

    final_train_data = to_dataset(final_train)
    final_test_data = to_dataset(final_test, with_labels=False)
    transformers.set_seed(SEED)
    final_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_PATH, num_labels=1, dtype=torch.float32)
    final_args = TrainingArguments(
        output_dir=f"{OUT_DIR}/final_full_data", num_train_epochs=EPOCHS,
        learning_rate=LR, per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=2 * BATCH_SIZE, warmup_ratio=0.1,
        weight_decay=0.01, eval_strategy="no", save_strategy="no",
        logging_steps=100, fp16=torch.cuda.is_available(), report_to="none", seed=SEED)
    final_trainer = Trainer(
        model=final_model, args=final_args, train_dataset=final_train_data,
        data_collator=DataCollatorWithPadding(tokenizer))
    final_started = __import__("time").perf_counter()
    final_trainer.train()
    final_fit_minutes = (__import__("time").perf_counter() - final_started) / 60
    final_test_pred = final_trainer.predict(final_test_data).predictions.reshape(-1)
    del final_trainer, final_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

submission = pd.DataFrame({"id": test["id"], "score": np.clip(final_test_pred, 0, 1)})
assert list(submission.columns) == ["id", "score"]
assert len(submission) == len(test) and submission["score"].notna().all()
submission.to_csv(f"{OUT_DIR}/submission.csv", index=False)
print(f"Selected final setting by validation Pearson: {best_setting}")
print(f"Final full-training time: {final_fit_minutes:.2f} minutes")
submission.head()

## 11. Validation error analysis

This analyzes the selected model's predictions on the same held-out validation rows. Pearson is undefined for sections with fewer than two rows or constant values.

In [ ]:
analysis = val.copy()
analysis["section"] = analysis["context"].astype(str).str[0]
analysis["abs_error"] = (analysis["pred"] - analysis["score"]).abs()


def section_pearson(group):
    if len(group) < 2 or group["score"].nunique() < 2 or group["pred"].nunique() < 2:
        return float("nan")
    return float(np.corrcoef(group["score"], group["pred"])[0, 1])


section_report = analysis.groupby("section").apply(
    lambda group: pd.Series({"rows": len(group), "Pearson": section_pearson(group)}),
    include_groups=False)
section_report = section_report.reindex(list("ABCDEFGH"))
display(section_report)


score_report = analysis.groupby("score")["pred"].agg(["count", "mean"]).rename(columns={"mean": "mean prediction"})
display(score_report)


worst_errors = analysis.nlargest(10, "abs_error")[
    ["id", "anchor", "target", "context", "score", "pred", "abs_error"]]
display(worst_errors)